# 03 · Multi-Model Training


Trains five diverse models — Logistic Regression, Random Forest, XGBoost, 
Linear Regression (as a linear score estimator), and LightGBM — on the same 
leakage-free training set. Each model is trained independently, evaluated on 
the 2024-2025 holdout test set, and saved for comparison in Notebook 05.


In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    precision_score, recall_score, f1_score, accuracy_score,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

DATA_DIR = Path("../../data_pipeline/data/processed")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Load artifacts from Notebook 02
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")["target_risk"]
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_test = pd.read_csv(DATA_DIR / "y_test.csv")["target_risk"]
scaler = joblib.load(MODEL_DIR / "scaler.joblib")
feature_cols = json.load(open(MODEL_DIR / "feature_columns.json"))

X_train = X_train[feature_cols]
X_test = X_test[feature_cols]
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Risk ratio — train: {y_train.mean():.2%} | test: {y_test.mean():.2%}")

# Scale for linear models
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Calculate scale_pos_weight for boosted models
scale_w = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_w:.2f}")

# ─── Model Definitions ───
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=5,
        class_weight="balanced", random_state=42, n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        scale_pos_weight=scale_w, eval_metric="logloss",
        random_state=42, use_label_encoder=False,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        scale_pos_weight=scale_w, random_state=42, verbose=-1,
    ),
}

# Linear Regression as a linear score estimator (regression, then threshold)
from sklearn.linear_model import LinearRegression
models["Linear Regression"] = LinearRegression()

# ─── Train All Models ───
results = {}
for name, model in models.items():
    print(f"\nTraining {name}...")
    if name == "Linear Regression":
        model.fit(X_train_scaled, y_train.astype(float))
    else:
        model.fit(X_train, y_train)
    results[name] = model
    print(f"  ✅ {name} trained")

# ─── Evaluate All Models ───
print("\n" + "=" * 80)
print("EVALUATION RESULTS")
print("=" * 80)

evaluation = {}
for name, model in results.items():
    print(f"\n--- {name} ---")
    is_lr_model = name == "Linear Regression"

    if is_lr_model:
        # Regression metrics
        y_pred_proba = model.predict(X_test_scaled)
        y_pred_proba = np.clip(y_pred_proba, 0, 1)
        y_pred = (y_pred_proba >= 0.5).astype(int)

        mse = np.mean((y_test.astype(float) - y_pred_proba) ** 2)
        r2 = 1 - np.sum((y_test.astype(float) - y_pred_proba) ** 2) / np.sum((y_test.astype(float) - y_test.mean()) ** 2)

        print(f"  MSE: {mse:.4f}")
        print(f"  R²: {r2:.4f}")
        print(classification_report(y_test, y_pred, target_names=["Safe (0)", "Risk (1)"]))

        evaluation[name] = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, y_pred_proba),
            "mse": mse,
            "r2": r2,
        }
    else:
        y_pred = model.predict(X_test if not name == "Logistic Regression" else X_test_scaled)
        y_pred_proba = model.predict_proba(X_test if not name == "Logistic Regression" else X_test_scaled)[:, 1]

        print(classification_report(y_test, y_pred, target_names=["Safe (0)", "Risk (1)"]))
        roc_auc = roc_auc_score(y_test, y_pred_proba)
        print(f"  ROC-AUC: {roc_auc:.4f}")

        evaluation[name] = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc,
        }

# ─── Confusion Matrices ───
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for i, (name, model) in enumerate(results.items()):
    is_lr_model = name == "Linear Regression"
    if is_lr_model:
        y_pred_local = (model.predict(X_test_scaled)).clip(0, 1)
        y_pred_local = (y_pred_local >= 0.5).astype(int)
    else:
        y_pred_local = model.predict(X_test if not name == "Logistic Regression" else X_test_scaled)
    cm = confusion_matrix(y_test, y_pred_local)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i],
                xticklabels=["Safe (0)", "Risk (1)"],
                yticklabels=["Safe (0)", "Risk (1)"])
    axes[i].set_title(name)
    axes[i].set_ylabel("Actual")
    axes[i].set_xlabel("Predicted")
plt.tight_layout()
plt.savefig(MODEL_DIR / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n💾 Confusion matrices saved to models/confusion_matrices.png")

# ─── Feature Importance (tree-based models) ───
tree_models = {n: m for n, m in results.items() if n not in ("Logistic Regression", "Linear Regression")}
if tree_models:
    fig, axes = plt.subplots(1, len(tree_models), figsize=(6 * len(tree_models), 6))
    if len(tree_models) == 1:
        axes = [axes]
    for ax, (name, model) in zip(axes, tree_models.items()):
        importances = model.feature_importances_
        feat_imp = pd.DataFrame({"feature": X_train.columns, "importance": importances})
        feat_imp = feat_imp.sort_values("importance", ascending=True)
        ax.barh(feat_imp["feature"], feat_imp["importance"], color="#2e7d32")
        ax.set_title(f"{name}\nFeature Importance")
        ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig(MODEL_DIR / "feature_importance.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("💾 Feature importance plots saved to models/feature_importance.png")

# ─── Save All Models ───
for name, model in results.items():
    safe_name = name.lower().replace(" ", "_")
    path = MODEL_DIR / f"{safe_name}.joblib"
    joblib.dump(model, path)
    print(f"💾 {name} saved to models/{safe_name}.joblib")

# ─── Save Evaluation Results ───
eval_df = pd.DataFrame(evaluation).T
eval_df.to_csv(MODEL_DIR / "model_evaluation.csv")
print(f"\n💾 Evaluation results saved to models/model_evaluation.csv")
print("\n" + "=" * 80)
print("TRAINING COMPLETE — All 5 models trained, evaluated, and saved.")
print("=" * 80)
print(eval_df.round(4).to_string())
